# Data Preparation: Convert Complex Portal TSV to Graph and Hypergraph
**Instructions:**
1. Run this cell by clicking the Play button on the left.
2. A "Choose Files" button will appear. Click it and select your `complex.tsv` file.
3. The script will process the file and automatically download the `complexportal_dataset.graph` and `complexportal_dataset.hypergraph` files to your computer.

# Data Preparation: Convert Complex Portal TSV to Graph, Hypergraph, and ID Map
**Instructions:**
1. Run this cell by clicking the Play button on the left.
2. A "Choose Files" button will appear. Click it and select your `complex.tsv` file.
3. The script will process the file and automatically download three files:
 - `complexportal_dataset.graph` (the pairwise graph)
 - `complexportal_dataset.hypergraph` (the hypergraph)
 - `complexportal_dataset_id_mapping.tsv` (the ID lookup table)



In [1]:
import os
import itertools
from google.colab import files

# --- Configuration ---
# This is the name of the file you will upload
COMPLEX_PORTAL_INPUT_FILE = 'complex.tsv'

# Base name for the output files
DATASET_BASE_NAME = 'complexportal_dataset'

# --- Output filenames with desired extensions ---
GRAPH_OUTPUT_FILE = f'{DATASET_BASE_NAME}.graph'
HYPERGRAPH_OUTPUT_FILE = f'{DATASET_BASE_NAME}.hypergraph'
MAPPING_OUTPUT_FILE = f'{DATASET_BASE_NAME}_id_mapping.tsv'

# --- Step 1: Upload the dataset ---
print(f"Please upload your '{COMPLEX_PORTAL_INPUT_FILE}' file.")
uploaded = files.upload()

if COMPLEX_PORTAL_INPUT_FILE not in uploaded:
    print(f"\nERROR: File upload failed or the uploaded file was not named '{COMPLEX_PORTAL_INPUT_FILE}'.")
    print("Please rename your file and try again.")
else:
    print(f"\nFile '{COMPLEX_PORTAL_INPUT_FILE}' uploaded successfully!")

    # --- Data Structures to hold processed data ---
    # Maps a unique molecule identifier (e.g., a UniProt ID) to a unique integer ID
    participant_to_id = {}
    next_participant_id = 0
    # A list of lists, where each inner list represents a complex using integer IDs
    all_complexes_as_ids = []

    # --- Step 2. Parse the Complex Portal TSV and Create Participant-ID Mapping ---
    print(f"\nReading and processing input file: {COMPLEX_PORTAL_INPUT_FILE}...")
    try:
        with open(COMPLEX_PORTAL_INPUT_FILE, 'r') as f:
            # Skip the header line which starts with '#'
            header = next(f)

            for line_num, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue

                parts = line.split('\t')
                # The participant list is in the 5th column (index 4)
                if len(parts) < 5:
                    print(f"Warning: Skipping line {line_num + 2} with fewer than 5 columns.")
                    continue

                participant_str = parts[4] # e.g., "P84022(1)|Q13485(1)|Q15796(1)"
                participants_in_complex = participant_str.split('|')

                current_complex_ids = []

                for participant_with_stoich in participants_in_complex:
                    # Extract only the identifier, removing stoichiometry like "(1)" or "(0)"
                    participant_symbol = participant_with_stoich.split('(')[0]

                    if participant_symbol not in participant_to_id:
                        participant_to_id[participant_symbol] = next_participant_id
                        next_participant_id += 1
                    current_complex_ids.append(participant_to_id[participant_symbol])

                if current_complex_ids:
                    unique_sorted_ids = sorted(list(set(current_complex_ids)))
                    all_complexes_as_ids.append(unique_sorted_ids)

    except Exception as e:
        print(f"An error occurred during file processing: {e}")

    print(f"Processed {len(all_complexes_as_ids)} complexes and found {len(participant_to_id)} unique participants (proteins, RNA, etc.).")

    # --- Step 3. Generate .graph File (Pairwise Edges / Clique Model) ---
    print(f"\nGenerating graph file: {GRAPH_OUTPUT_FILE}...")
    pairwise_edges = set()
    for complex_ids in all_complexes_as_ids:
        if len(complex_ids) < 2:
            continue
        for u, v in itertools.combinations(complex_ids, 2):
            pairwise_edges.add(tuple(sorted((u, v))))

    num_nodes_graph = len(participant_to_id)
    num_edges_graph = len(pairwise_edges)

    with open(GRAPH_OUTPUT_FILE, 'w') as f:
        f.write(f"{num_nodes_graph} {num_edges_graph}\n")
        for u, v in sorted(list(pairwise_edges)):
            f.write(f"{u} {v}\n")

    print(f"Graph file generated: {num_nodes_graph} nodes, {num_edges_graph} edges.")

    # --- Step 4. Generate .hypergraph File (Complexes as Hyperedges) ---
    print(f"Generating hypergraph file: {HYPERGRAPH_OUTPUT_FILE}...")
    with open(HYPERGRAPH_OUTPUT_FILE, 'w') as f:
        for complex_ids in all_complexes_as_ids:
            if complex_ids:
                f.write(" ".join(map(str, complex_ids)) + "\n")

    print(f"Hypergraph file generated: {len(all_complexes_as_ids)} hyperedges.")

    # --- Step 5. Generate ID Mapping File ---
    print(f"Generating ID mapping file: {MAPPING_OUTPUT_FILE}...")
    # Reverse the dictionary for easy lookup from integer ID back to the participant symbol
    id_to_participant = {v: k for k, v in participant_to_id.items()}

    with open(MAPPING_OUTPUT_FILE, 'w') as f:
        f.write("Integer_ID\tParticipant_ID\n") # Write a helpful header
        # Sort by the integer ID for a clean, ordered file
        for int_id in sorted(id_to_participant.keys()):
            participant_symbol = id_to_participant[int_id]
            f.write(f"{int_id}\t{participant_symbol}\n")

    print(f"ID mapping file generated.")

    # --- Step 6. Download all generated files ---
    print("\nTriggering file downloads...")
    files.download(GRAPH_OUTPUT_FILE)
    files.download(HYPERGRAPH_OUTPUT_FILE)
    files.download(MAPPING_OUTPUT_FILE)
    print("\nAll done!")

Please upload your 'complex.tsv' file.


Saving complex.tsv to complex.tsv

File 'complex.tsv' uploaded successfully!

Reading and processing input file: complex.tsv...
Processed 2346 complexes and found 3693 unique participants (proteins, RNA, etc.).

Generating graph file: complexportal_dataset.graph...
Graph file generated: 3693 nodes, 19662 edges.
Generating hypergraph file: complexportal_dataset.hypergraph...
Hypergraph file generated: 2346 hyperedges.
Generating ID mapping file: complexportal_dataset_id_mapping.tsv...
ID mapping file generated.

Triggering file downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All done!
